# Drug Dataset — NLP Text Cleaning Pipeline

This notebook cleans the `drugs_data.parquet` dataset through a structured NLP pipeline:

1. Load & inspect
2. Strip section header boilerplate
3. Normalize casing
4. Remove special characters & Unicode artifacts
5. Deduplicate repeated paragraphs within cells
6. Handle high-missingness columns
7. Normalize the `route` column
8. Deduplicate brand+generic pairs
9. Export cleaned dataset

## 0. Install Dependencies

In [1]:
# Install required packages (run once)
!pip install pyarrow pandas --quiet


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## 1. Load & Inspect

In [2]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_parquet('drugs_data.parquet')

print(f'Shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nDtypes:\n{df.dtypes}')

Shape: (42481, 10)

Columns: ['brand_name', 'generic_name', 'route', 'indications', 'dosage', 'contraindications', 'side_effects', 'warnings', 'do_not_use', 'use_when']

Dtypes:
brand_name           str
generic_name         str
route                str
indications          str
dosage               str
contraindications    str
side_effects         str
warnings             str
do_not_use           str
use_when             str
dtype: object


In [3]:
# Null counts and percentages
null_summary = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct':   (df.isnull().mean() * 100).round(1)
})
print('Null Summary:')
print(null_summary.to_string())

Null Summary:
                   null_count  null_pct
brand_name                  3       0.0
generic_name                3       0.0
route                    1008       2.4
indications               196       0.5
dosage                    342       0.8
contraindications       34735      81.8
side_effects            34542      81.3
warnings                 5840      13.7
do_not_use              26621      62.7
use_when                25500      60.0


In [4]:
# Preview first 3 rows
df.head(3)

,brand_name,generic_name,route,indications,dosage,contraindications,side_effects,warnings,do_not_use,use_when
0,SILICEA,SILICEA,ORAL,INDICATIONS: May temporarily relieve unhealthy...,": Adults and children 5 to 10 drops orally, 1 ...",NaN,NaN,This product is to be used for self-limiting c...,if capseal is broken or missing. Close the cap...,NaN
1,RUTA GRAVEOLENS,RUTA GRAVEOLENS,ORAL,INDICATIONS Condition listed above or as direc...,"DOSAGE Adults- Take 4 or 6 Pellets by mouth, t...",NaN,NaN,This product is to be used for self-limiting c...,if capseal is broken or missing. Close the cap...,NaN
2,Tryptophan,L-TRYPTOPHAN,ORAL,"INDICATIONS: For temporary relief of anxiety, ...",": 10 drops orally, 3 times a day. Consult a ph...",NaN,NaN,": ​If pregnant or breast-feeding, ​ ask a heal...",NaN,NaN


## 2. Strip Section Header Boilerplate

Each text column contains embedded section labels (`"INDICATIONS: ..."`, `"4 CONTRAINDICATIONS ..."`, `"DOSAGE Adults-"`) that are noise for downstream NLP tasks.

In [5]:
# Map each column to the section header keywords likely to appear in it
HEADER_PATTERNS = {
    'indications':       r'^[\d\s]*\b(INDICATIONS?|USES?|USE|HOMEOPATHIC USES?)\b[\s:\-]*',
    'dosage':            r'^[\d\s]*\b(DOSAGE(\s+AND\s+ADMINISTRATION)?|DIRECTIONS?)\b[\s:\-]*',
    'contraindications': r'^[\d\s]*\b(CONTRAINDICATIONS?)\b[\s:\-]*',
    'side_effects':      r'^[\d\s]*\b(ADVERSE\s+REACTIONS?|SIDE\s+EFFECTS?)\b[\s:\-]*',
    'warnings':          r'^[\d\s]*\b(WARNINGS?(\s+AND\s+PRECAUTIONS?)?)\b[\s:\-]*',
    'do_not_use':        r'^[\d\s]*\b(DO\s+NOT\s+USE|CONTRAINDICATIONS?)\b[\s:\-]*',
    'use_when':          r'^[\d\s]*\b(WHEN\s+USING|USE\s+WHEN)\b[\s:\-]*',
}

def strip_section_header(text, pattern):
    if pd.isna(text):
        return text
    cleaned = re.sub(pattern, '', str(text), flags=re.IGNORECASE).strip()
    # Also strip a leading colon/dash that may remain after the header
    cleaned = re.sub(r'^[:\-–—]\s*', '', cleaned)
    return cleaned if cleaned else text  # fallback to original if result is empty

df_clean = df.copy()

for col, pattern in HEADER_PATTERNS.items():
    if col in df_clean.columns:
        before = df_clean[col].dropna().str[:40].tolist()[:2]
        df_clean[col] = df_clean[col].apply(lambda x: strip_section_header(x, pattern))
        after  = df_clean[col].dropna().str[:40].tolist()[:2]
        print(f'[{col}]')
        print(f'  Before: {before}')
        print(f'  After:  {after}\n')

[indications]
  Before: ['INDICATIONS: May temporarily relieve unh', 'INDICATIONS Condition listed above or as']
  After:  ['May temporarily relieve unhealthy skin w', 'Condition listed above or as directed by']

[dosage]
  Before: [': Adults and children 5 to 10 drops oral', 'DOSAGE Adults- Take 4 or 6 Pellets by mo']
  After:  ['Adults and children 5 to 10 drops orally', 'Adults- Take 4 or 6 Pellets by mouth, th']

[contraindications]
  Before: ['4 CONTRAINDICATIONS Do not use in childr', 'Contrainidcations CONTRAINDICATIONS: Chl']
  After:  ['Do not use in children under 12 years of', 'Contrainidcations CONTRAINDICATIONS: Chl']

[side_effects]
  Before: ['6 ADVERSE REACTIONS Allergic reactions a', 'ADVERSE REACTIONS: The most common side ']
  After:  ['Allergic reactions and other idiosyncras', 'The most common side effects associated ']

[warnings]
  Before: ['This product is to be used for self-limi', 'This product is to be used for self-limi']
  After:  ['This product is to be us

## 3. Normalize Casing

`brand_name` and `generic_name` are inconsistently cased — some ALL CAPS, some Title Case. Standardize to Title Case.

In [6]:
print('Before:')
print(df_clean[['brand_name', 'generic_name']].head(6).to_string())

df_clean['brand_name']   = df_clean['brand_name'].str.strip().str.title()
df_clean['generic_name'] = df_clean['generic_name'].str.strip().str.title()
df_clean['route']        = df_clean['route'].str.strip().str.upper()

print('\nAfter:')
print(df_clean[['brand_name', 'generic_name']].head(6).to_string())

Before:
           brand_name        generic_name
0             SILICEA             SILICEA
1     RUTA GRAVEOLENS     RUTA GRAVEOLENS
2          Tryptophan        L-TRYPTOPHAN
3       KALI BROMATUM       KALI BROMATUM
4  CALCAREA CARBONICA  CALCAREA CARBONICA
5        STAPHYSAGRIA        STAPHYSAGRIA

After:
           brand_name        generic_name
0             Silicea             Silicea
1     Ruta Graveolens     Ruta Graveolens
2          Tryptophan        L-Tryptophan
3       Kali Bromatum       Kali Bromatum
4  Calcarea Carbonica  Calcarea Carbonica
5        Staphysagria        Staphysagria


## 4. Remove Special Characters & Unicode Artifacts

Issues found:
- **510** rows with zero-width spaces (`\u200b`) in `warnings`
- **891** rows with markdown bold markers (`**...**`) in `indications`
- **3,979** rows with `[see Warnings and Precautions (5.1)]`-style cross-reference anchors in `side_effects`
- Non-breaking spaces (`\u00a0`)

In [7]:
def clean_special_chars(text):
    if pd.isna(text):
        return text
    t = str(text)
    # Unicode artifacts
    t = t.replace('\u200b', ' ')          # zero-width space
    t = t.replace('\u00a0', ' ')          # non-breaking space
    t = t.replace('\u2019', "'")          # right single quotation mark
    t = t.replace('\u2018', "'")          # left single quotation mark
    t = t.replace('\u201c', '"')          # left double quotation mark
    t = t.replace('\u201d', '"')          # right double quotation mark
    t = t.replace('\u2013', '-')          # en dash
    t = t.replace('\u2014', '-')          # em dash
    # Markdown bold/italic markers
    t = re.sub(r'\*{1,3}', '', t)
    # Cross-reference anchors like [see Warnings and Precautions (5.1)]
    t = re.sub(r'\[see [^\]]+\]', '', t, flags=re.IGNORECASE)
    # Numbered section references like (5.1), (6.1)
    t = re.sub(r'\(\s*\d+\.\d+\s*\)', '', t)
    # Extra whitespace
    t = re.sub(r'[ \t]{2,}', ' ', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    return t.strip()

TEXT_COLS = ['indications', 'dosage', 'contraindications', 'side_effects',
             'warnings', 'do_not_use', 'use_when']

for col in TEXT_COLS:
    df_clean[col] = df_clean[col].apply(clean_special_chars)

print('Special character cleaning applied to all text columns.')

# Verify zero-width spaces are gone
remaining_zws = df_clean['warnings'].astype(str).str.contains('\u200b').sum()
print(f'Remaining zero-width spaces in warnings: {remaining_zws}')

Special character cleaning applied to all text columns.
Remaining zero-width spaces in warnings: 0


## 5. Deduplicate Repeated Paragraphs Within Cells

`side_effects` and `warnings` fields contain entire paragraphs repeated verbatim — a copy-paste artifact from FDA label sources. This is removed by sentence-level deduplication.

In [8]:
def dedup_paragraphs(text):
    """Remove duplicate sentences/paragraphs within a single text field."""
    if pd.isna(text):
        return text
    t = str(text)
    # Split on sentence-ending punctuation or double newlines
    segments = re.split(r'(?<=[.!?])\s+|\n{2,}', t)
    seen = set()
    unique_segments = []
    for seg in segments:
        key = re.sub(r'\s+', ' ', seg.strip().lower())
        if key and key not in seen:
            seen.add(key)
            unique_segments.append(seg.strip())
    return ' '.join(unique_segments)

# Apply to columns known to have repeated content
for col in ['side_effects', 'warnings', 'contraindications']:
    before_len = df_clean[col].dropna().str.len().mean()
    df_clean[col] = df_clean[col].apply(dedup_paragraphs)
    after_len  = df_clean[col].dropna().str.len().mean()
    reduction  = (1 - after_len / before_len) * 100
    print(f'[{col}] Avg length: {before_len:.0f} → {after_len:.0f} chars ({reduction:.1f}% reduction)')

[side_effects] Avg length: 5927 → 5735 chars (3.3% reduction)
[warnings] Avg length: 817 → 798 chars (2.2% reduction)
[contraindications] Avg length: 484 → 478 chars (1.3% reduction)


## 6. Handle High-Missingness Columns

Several columns are majority null. Strategy:
- Merge `do_not_use` into `warnings` (they are semantically related)
- Merge `use_when` into `indications` (context for use)
- Fill remaining nulls with `"Not specified"` so rows are preserved

In [9]:
# ── Merge do_not_use → warnings ────────────────────────────────────────────
def merge_fields(base, extra, separator=' | DO NOT USE: '):
    if pd.isna(extra) or str(extra).strip() == '':
        return base
    if pd.isna(base) or str(base).strip() == '':
        return str(extra)
    return str(base) + separator + str(extra)

df_clean['warnings'] = df_clean.apply(
    lambda r: merge_fields(r['warnings'], r['do_not_use']), axis=1
)

# ── Merge use_when → indications ──────────────────────────────────────────
df_clean['indications'] = df_clean.apply(
    lambda r: merge_fields(r['indications'], r['use_when'], separator=' | USE WHEN: '), axis=1
)

# Drop the now-merged source columns
df_clean = df_clean.drop(columns=['do_not_use', 'use_when'])
print('Dropped: do_not_use, use_when (merged into warnings and indications)')

# ── Add a presence indicator for sparse columns ────────────────────────────
df_clean['has_contraindications'] = df_clean['contraindications'].notna().astype(int)
df_clean['has_side_effects']      = df_clean['side_effects'].notna().astype(int)
print('Added binary indicator columns: has_contraindications, has_side_effects')

# ── Fill remaining nulls ──────────────────────────────────────────────────
fill_cols = ['indications', 'dosage', 'contraindications', 'side_effects', 'warnings']
df_clean[fill_cols] = df_clean[fill_cols].fillna('Not specified')

print('\nNull counts after handling:')
print(df_clean.isnull().sum())

Dropped: do_not_use, use_when (merged into warnings and indications)
Added binary indicator columns: has_contraindications, has_side_effects

Null counts after handling:
brand_name                  3
generic_name                3
route                    1008
indications                 0
dosage                      0
contraindications           0
side_effects                0
warnings                    0
has_contraindications       0
has_side_effects            0
dtype: int64


## 7. Normalize the `route` Column

Route values are inconsistent. Map them to a controlled vocabulary.

In [10]:
print('Raw route value counts:')
print(df_clean['route'].value_counts().head(20))

Raw route value counts:
route
TOPICAL                     19271
ORAL                        15889
INTRAVENOUS                  1244
DENTAL                        839
OPHTHALMIC                    832
SUBLINGUAL                    675
NASAL                         469
INTRAMUSCULAR                 407
SUBCUTANEOUS                  311
CUTANEOUS                     289
RECTAL                        198
RESPIRATORY (INHALATION)      193
TRANSDERMAL                   144
VAGINAL                       127
AURICULAR (OTIC)              111
EPIDURAL                       69
EXTRACORPOREAL                 47
INFILTRATION                   42
BUCCAL                         38
PERCUTANEOUS                   26
Name: count, dtype: int64


In [11]:
ROUTE_MAP = {
    'ORAL':                        'oral',
    'TOPICAL':                     'topical',
    'OPHTHALMIC':                  'ophthalmic',
    'INTRAVENOUS':                 'intravenous',
    'SUBCUTANEOUS':                'subcutaneous',
    'INTRAMUSCULAR':               'intramuscular',
    'NASAL':                       'nasal',
    'RECTAL':                      'rectal',
    'VAGINAL':                     'vaginal',
    'TRANSDERMAL':                 'transdermal',
    'INHALATION':                  'inhalation',
    'AURICULAR (OTIC)':            'otic',
    'DENTAL':                      'dental',
    'INTRATHECAL':                 'intrathecal',
    'INTRAARTICULAR':              'intraarticular',
    'SUBLINGUAL':                  'sublingual',
    'BUCCAL':                      'buccal',
}

df_clean['route'] = (
    df_clean['route']
    .str.strip()
    .str.upper()
    .map(ROUTE_MAP)
    .fillna('unknown')
)

print('Normalized route value counts:')
print(df_clean['route'].value_counts())

Normalized route value counts:
route
topical          19271
oral             15889
unknown           1911
intravenous       1244
dental             839
ophthalmic         832
sublingual         675
nasal              469
intramuscular      407
subcutaneous       311
rectal             198
transdermal        144
vaginal            127
otic               111
buccal              38
intrathecal         15
Name: count, dtype: int64


## 8. Deduplicate brand+generic Pairs

**608** rows share the same `brand_name` + `generic_name`. Keep the most informative row (the one with the most non-null / non-empty fields).

In [12]:
print(f'Rows before deduplication: {len(df_clean)}')
print(f'Duplicate brand+generic pairs: {df_clean.duplicated(subset=["brand_name","generic_name"]).sum()}')

# Score each row by number of non-null, non-empty, non-"Not specified" fields
text_quality_cols = ['indications', 'dosage', 'contraindications', 'side_effects', 'warnings']

df_clean['_quality_score'] = df_clean[text_quality_cols].apply(
    lambda row: sum(
        1 for v in row
        if pd.notna(v) and str(v).strip() not in ('', 'Not specified')
    ),
    axis=1
)

# Keep the highest-quality row per brand+generic pair
df_clean = (
    df_clean
    .sort_values('_quality_score', ascending=False)
    .drop_duplicates(subset=['brand_name', 'generic_name'], keep='first')
    .drop(columns=['_quality_score'])
    .reset_index(drop=True)
)

print(f'\nRows after deduplication: {len(df_clean)}')

Rows before deduplication: 42481
Duplicate brand+generic pairs: 3247

Rows after deduplication: 39234


## 9. Final Inspection

In [13]:
print('=== Final Dataset Summary ===')
print(f'Shape: {df_clean.shape}')
print(f'\nColumns: {df_clean.columns.tolist()}')
print(f'\nNull counts:\n{df_clean.isnull().sum()}')
print(f'\nRoute distribution:\n{df_clean["route"].value_counts()}')

=== Final Dataset Summary ===
Shape: (39234, 10)

Columns: ['brand_name', 'generic_name', 'route', 'indications', 'dosage', 'contraindications', 'side_effects', 'warnings', 'has_contraindications', 'has_side_effects']

Null counts:
brand_name               3
generic_name             3
route                    0
indications              0
dosage                   0
contraindications        0
side_effects             0
warnings                 0
has_contraindications    0
has_side_effects         0
dtype: int64

Route distribution:
route
topical          18876
oral             14021
unknown           1725
intravenous        911
dental             826
ophthalmic         753
sublingual         651
nasal              422
subcutaneous       265
intramuscular      236
rectal             173
transdermal        126
vaginal            111
otic                93
buccal              33
intrathecal         12
Name: count, dtype: int64


In [14]:
# Quick before/after text comparison for a sample row
idx = 0
print('=== Sample Row — indications ===')
print('ORIGINAL:', repr(df['indications'].iloc[idx]))
print()
print('CLEANED: ', repr(df_clean['indications'].iloc[idx]))

=== Sample Row — indications ===
ORIGINAL: 'INDICATIONS: May temporarily relieve unhealthy skin with difficulty healing even small wounds.** **Claims based on traditional homeopathic practice, not accepted medical evidence. Not FDA evaluated.'

CLEANED:  'Hyoscyamine sulfate is effective as adjunctive therapy in the treatment of peptic ulcer. It can also be used to control gastric secretion, visceral spasm and hypermotility in spastic colitis, spastic bladder, cystitis, pylorospasm, and associated abdominal cramps. May be used in functional intestinal disorders to reduce symptoms such as those seen in mild dysenteries, diverticulitis, and acute enterocolitis. For use as adjunctive therapy in the treatment of irritable bowel syndrome (irritable colon, spastic colon, mucous colitis) and functional gastrointestinal disorders. Also used as adjunctive therapy in the treatment of neurogenic bladder and neurogenic bowel disturbances (including the splenic flexure syndrome and neurogenic colon

In [15]:
# Text length distributions after cleaning
print('Avg text lengths (cleaned):')
for col in ['indications', 'dosage', 'warnings', 'side_effects', 'contraindications']:
    lengths = df_clean[col].astype(str).str.len()
    print(f'  {col:20s}: mean={lengths.mean():.0f}, median={lengths.median():.0f}, max={lengths.max()}')

Avg text lengths (cleaned):
  indications         : mean=357, median=215, max=22499
  dosage              : mean=881, median=262, max=55015
  warnings            : mean=694, median=319, max=36917
  side_effects        : mean=861, median=13, max=168076
  contraindications   : mean=78, median=13, max=13523


## 10. Export Cleaned Dataset

In [16]:
output_path = 'drugs_data_cleaned.parquet'
df_clean.to_parquet(output_path, index=False)
print(f'Saved cleaned dataset → {output_path}')
print(f'Final shape: {df_clean.shape}')

Saved cleaned dataset → drugs_data_cleaned.parquet
Final shape: (39234, 10)


In [18]:
# Verify the saved file loads correctly
df_verify = pd.read_parquet(output_path)
print(f'Verified shape: {df_verify.shape}')
df_verify.head(10)

Verified shape: (39234, 10)


,brand_name,generic_name,route,indications,dosage,contraindications,side_effects,warnings,has_contraindications,has_side_effects
0,Hyoscyamine Sulfate,Hyoscyamine Sulfate,oral,Hyoscyamine sulfate is effective as adjunctive...,may be adjusted according to the conditions an...,"Glaucoma; obstructive uropathy (for example, b...",All of the following adverse reactions have be...,In the presence of high environmental temperat...,1,1
1,Cyclosporine,Cyclosporine,oral,"Kidney, Liver, and Heart Transplantation Cyclo...",Cyclosporine capsules (modified) has increased...,General Cyclosporine capsules (modified) is co...,"Kidney, Liver, and Heart Transplantation The p...",(See also BOXED WARNING ) All Patients Cyclosp...,1,1
2,Timolol Maleate,Timolol Maleate,ophthalmic,Timolol maleate ophthalmic gel forming solutio...,Patients should be instructed to invert the cl...,Timolol maleate ophthalmic gel forming solutio...,"In clinical trials, transient blurred vision u...",As with many topically applied ophthalmic drug...,1,1
3,Nitrofurantoin (Macrocrystals),Nitrofurantoin (Macrocrystals),oral,Nitrofurantoin capsules (macrocrystals) are sp...,Nitrofurantoin capsules (macrocrystals) should...,"Anuria, oliguria, or significant impairment of...","Respiratory: CHRONIC, SUBACUTE, OR ACUTE PULMO...","Pulmonary reactions: ACUTE, SUBACUTE, OR CHRON...",1,1
4,Azithromycin,Azithromycin Monohydrate,intravenous,AND USAGE Azithromycin for injection is a macr...,(See INDICATIONS AND USAGE and CLINICAL PHARMA...,Patients with known hypersensitivity to azithr...,In clinical trials of intravenous azithromycin...,"Hypersensitivity Serious allergic reactions, i...",1,1
5,Clindamycin,Clindamycin Phosphate,intramuscular,"Clindamycin Injection, USP is indicated in the...","If diarrhea occurs during therapy, this antibi...",This drug is contraindicated in individuals wi...,The following reactions have been reported wit...,See BOXED WARNING . Clostridioides difficile -...,1,1
6,Neobenz Micro Plus Pack,Benzoyl Peroxide,unknown,"III. INDICATIONS AND USAGE NeoBenz® Micro, Neo...","IX. DOSAGE AND ADMINISTRATION NeoBenz® Micro, ...","IV. CONTRAINDICATIONS NeoBenz® Micro, NeoBenz®...",VII . ADVERSE REACTIONS Allergic contact derma...,"V. WARNINGS When using this product, avoid unn...",1,1
7,Cefazolin,Cefazolin Sodium,intramuscular,"Cefazolin for Injection, USP is indicated for ...",Usual Adult Dosage Type of Infection Dose Freq...,CEFAZOLIN FOR INJECTION IS CONTRAINDICATED IN ...,The following reactions have been reported: Ga...,BEFORE THERAPY WITH CEFAZOLIN FOR INJECTION IS...,1,1
8,Oxytocin,Oxytocin,intramuscular,"IMPORTANT NOTICE: Oxytocin Injection, USP (syn...",of oxytocin is determined by uterine response....,Oxytocin injection (synthetic) is contraindica...,"To report SUSPECTED ADVERSE REACTIONS, contact...",Oxytocin injection (synthetic) when given for ...,1,1
9,Cupric Chloride,Cupric Chloride,intravenous,Cupric chloride injection 0.4 mg/mL is indicat...,"Cupric Chloride Injection, USP 0.4 mg/mL conta...",None known.,None known.,Direct intramuscular or intravenous injection ...,1,1


BEFORE THERAPY WITH CEFAZOLIN FOR INJECTION IS INSTITUTED, CAREFUL INQUIRY SHOULD BE MADE TO DETERMINE WHETHER THE PATIENT HAS HAD PREVIOUS HYPERSENSITIVITY REACTIONS TO CEFAZOLIN, CEPHALOSPORINS, PENICILLINS, OR OTHER DRUGS. IF THIS PRODUCT IS GIVEN TO PENICILLIN-SENSITIVE PATIENTS, CAUTION SHOULD BE EXERCISED BECAUSE CROSS-HYPERSENSITIVITY AMONG BETA-LACTAM ANTIBIOTICS HAS BEEN CLEARLY DOCUMENTED AND MAY OCCUR IN UP TO 10% OF PATIENTS WITH A HISTORY OF PENICILLIN ALLERGY. IF AN ALLERGIC REACTION TO CEFAZOLIN FOR INJECTION OCCURS, DISCONTINUE TREATMENT WITH THE DRUG. SERIOUS ACUTE HYPERSENSITIVITY REACTIONS MAY REQUIRE TREATMENT WITH EPINEPHRINE AND OTHER EMERGENCY MEASURES, INCLUDING OXYGEN, IV FLUIDS, IV ANTIHISTAMINES, CORTICOSTEROIDS, PRESSOR AMINES, AND AIRWAY MANAGEMENT, AS CLINICALLY INDICATED. Pseudomembranous colitis has been reported with nearly all antibacterial agents, including cefazolin, and may range in severity from mild to life-threatening. Therefore, it is important to consider this diagnosis in patients who present with diarrhea subsequent to the administration of antibacterial agents. Treatment with antibacterial agents alters the normal flora of the colon and may permit overgrowth of clostridia. Studies indicate that a toxin produced by Clostridium difficile is a primary cause of "antibiotic-associated colitis." After the diagnosis of pseudomembranous colitis has been established, therapeutic measures should be initiated. Mild cases of pseudomembranous colitis usually respond to drug discontinuation alone. In moderate to severe cases, consideration should be given to management with fluids and electrolytes, protein supplementation, and treatment with an oral antibacterial drug clinically effective against C. difficile colitis.

In [19]:
df[df['brand_name']=='Ibuprofen']

,brand_name,generic_name,route,indications,dosage,contraindications,side_effects,warnings,do_not_use,use_when
495,Ibuprofen,IBUPROFEN,ORAL,Carefully consider the potential benefits and ...,Carefully consider the potential benefits and ...,Ibuprofen tablets are contraindicated in patie...,The most frequent type of adverse reaction occ...,CARDIOVASCULAR EFFECTS Cardiovascular Thrombot...,if you have ever had an allergic reaction to a...,When using this product take with food or milk...
16341,Ibuprofen,IBUPROFEN CAPSULE,ORAL,Uses temporarily relieves minor aches and pain...,do not take more than directed the smallest ef...,NaN,NaN,Allergy alert: Ibuprofen may cause a severe al...,if you have ever had an allergic reaction to a...,When using this product take with food or milk...
18692,Ibuprofen,IBUPROFEN ORAL,ORAL,Carefully consider the potential benefits and ...,Carefully consider the potential benefits and ...,Ibuprofen oral suspension is contraindicated i...,"In patients taking ibuprofen or other NSAIDs, ...",CARDIOVASCULAR EFFECTS Cardiovascular Thrombot...,NaN,NaN
20058,Ibuprofen,"IBUPROFEN TABLETS USP, 200MG",ORAL,Uses temporarily relieves minor aches and pain...,do not take more than directed the smallest ef...,NaN,NaN,Allergy alert: Ibuprofen may cause a severe al...,if you have ever had an allergic reaction to i...,When using this product take with food or milk...
27876,Ibuprofen,IBUPROFEN CAPSULES 200 MG,ORAL,Uses temporarily relieves minor aches and pain...,do not take more than directed​ the smallest e...,NaN,NaN,Allergy alerts: Ibuprofen may cause a severe a...,if you have ever had an allergic reaction to a...,When using this product take with food or milk...
35802,Ibuprofen,"IBUPROFEN TABLET, FILM-COATED",ORAL,Uses Uses temporarily relieves minor aches and...,Directions do not take more than directed the ...,NaN,NaN,Alergy Alert Allergy alert: Ibuprofen may caus...,Do not use if you have ever had an allergic re...,When using this product When using this produc...
